# Tiếp tục A3 refit trên T4 Colab
Khôi phục từ checkpoint H100 đã kiểm trên D. TRAIN90,9epoch,batch128,FP32,seed42. Giữ confidence TRAIN80. Chạy từng cell theo thứ tự, chỉ một lượt training có hạn giờ. Cần sao lưu checkpoint mới về D trước khi trả runtime.

In [ ]:
import json, platform, shutil, subprocess, sys
from pathlib import Path
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10) if shutil.which("nvidia-smi") else None
print(json.dumps({"status":"DISCOVERY_ONLY", "python":sys.version, "platform":platform.platform(), "gpu":None if query is None else query.stdout.strip(), "gpu_query_code":None if query is None else query.returncode, "project_exists":Path("/opt/kltn").exists(), "free_disk_bytes":shutil.disk_usage("/content").free}))


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json
print("Chon COLAB_RESUME.zip tu thu muc model-export-a3-refit-recovery-20261115-02 tren o D.", flush=True)
uploaded = files.upload()
assert list(uploaded) == ["COLAB_RESUME.zip"], "Can dung mot file COLAB_RESUME.zip"
payload = uploaded.pop("COLAB_RESUME.zip")
assert len(payload) == 683393489
assert hashlib.sha256(payload).hexdigest() == "0cbcbc0d845e9eb9f0a68dac9b81d1c5c4ed9fdc3ddb557cc431fa8858ec1f16"
transfer = Path("/content/kltn-a3-transfer")
transfer.mkdir(exist_ok=True)
archive = transfer / "COLAB_RESUME.zip"
assert not archive.exists(), "Goi da ton tai, can kiem tra truoc khi ghi"
archive.write_bytes(payload)
del payload, uploaded
print(json.dumps({"status":"PASS_UPLOAD_SHA256", "path":str(archive), "bytes":archive.stat().st_size}))


In [ ]:
"""Kiểm và giải gói A3 trên Colab, giữ nguyên snapshot số học đã đóng."""
import hashlib
import json
from pathlib import Path
import shutil
import zipfile


def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def checked_destination(root, name):
    destination = (root / name).resolve()
    if not destination.is_relative_to(root.resolve()):
        raise ValueError('Member vượt thư mục đích')
    return destination


def copy_member(saved, member, destination, pin=None):
    if destination.exists():
        if pin and destination.stat().st_size == pin['bytes'] and digest(destination) == pin['sha256']:
            return
        raise FileExistsError(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with saved.open(member) as source, destination.open('xb') as target:
        shutil.copyfileobj(source, target)
    if pin and (destination.stat().st_size != pin['bytes'] or digest(destination) != pin['sha256']):
        raise ValueError('Member không khớp inventory ' + member)


def bootstrap(index):
    transfer = Path('/content/kltn-a3-transfer')
    archive = transfer / index['archive']
    assert archive.stat().st_size == index['archive_bytes']
    assert digest(archive) == index['archive_sha256']
    root = Path(index['project_root'])
    source_zip = transfer / 'SOURCE.zip'
    with zipfile.ZipFile(archive) as saved:
        expected = {'SOURCE.zip', *index['entries']}
        assert set(saved.namelist()) == expected
        assert len(saved.namelist()) == len(expected)
        copy_member(saved, 'SOURCE.zip', source_zip,
                    dict(bytes=saved.getinfo('SOURCE.zip').file_size, sha256=index['source_zip_sha256']))
        with zipfile.ZipFile(source_zip) as source:
            inventory = json.loads(source.read('INPUTS.json'))
            assert set(source.namelist()) == {'INPUTS.json', 'CPU_CHECK.json',
                                             *('project/' + name for name in inventory['files'])}
            assert json.loads(source.read('CPU_CHECK.json'))['status'] == 'PASS_LOCAL_STAGE_INPUTS'
            for name, pin in inventory['files'].items():
                copy_member(source, 'project/' + name, checked_destination(root, name), pin)
        for name, pin in index['entries'].items():
            assert name.startswith('recovery/')
            copy_member(saved, name, checked_destination(root, name.removeprefix('recovery/')), pin)
    assert digest(root / index['resume_checkpoint']) == index['resume_sha256']
    proof = dict(status='PASS_COLAB_A3_INPUT_SHA256', project_root=str(root),
                 source_files=len(inventory['files']), recovery_files=len(index['entries']),
                 checkpoint_sha256=index['resume_sha256'], complete_epochs=index['complete_epochs'],
                 partial_epoch=index['partial_epoch'], partial_optimizer_step=index['partial_optimizer_step'],
                 training_started=False)
    (transfer / 'bootstrap-verified.json').write_text(json.dumps(proof, indent=2))
    print(json.dumps(proof))

bootstrap({'schema': 'a3-portable-refit-recovery-v1', 'status': 'RESUMABLE_PARTIAL_REFIT', 'archive': 'COLAB_RESUME.zip', 'archive_bytes': 683393489, 'archive_sha256': '0cbcbc0d845e9eb9f0a68dac9b81d1c5c4ed9fdc3ddb557cc431fa8858ec1f16', 'source_zip_sha256': 'c3a109b4c07bcd7d802ccc35938a9d94be8d36377090fe08d6f1426ea523938a', 'project_root': '/opt/kltn', 'complete_epochs': 2, 'partial_epoch': 3, 'partial_optimizer_step': 1254, 'frozen_epochs': 9, 'resume_checkpoint': 'experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/refit-batch-epoch-3-step-1254-7581baf4e5594c54b52a6ac7df24daa9/state.pt', 'resume_sha256': '13ff97d60efabef65b84b03d178d8c2a24a4c8975febc687d1f9eca895c138e6', 'entries': {'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/attempt-884e5f7f62764b92934e6a3065f66cf5.json': {'bytes': 233, 'sha256': '1ed9443ca1ce094354d1b64c84bee252ac41153720d4c96b70b544aa2f8c944a'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/attempt-a5a52d307d314bd585f8250721c29e7e.json': {'bytes': 295, 'sha256': 'dae906372dc19e72070ca482200898f8da8d8a98a3a5666145ebd009b5d8d1ed'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-001/COMPLETE.json': {'bytes': 183, 'sha256': 'c6797784fc601ab90b7cb91d2b7e3d36d86776525be52ea840ca073b43e4054a'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-001/metadata.json': {'bytes': 2286, 'sha256': '7d68037dd267a49d35efb90ae218b45b9a0c168555fa8280e8bb017352e6d8f3'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-001/state.pt': {'bytes': 28913759, 'sha256': 'd0f60d41638bd9263275327f2740fc72b037a9938e9609755dc843cece9a9c34'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-002/COMPLETE.json': {'bytes': 183, 'sha256': 'ab14943816fc00286d5e3a002b9532f0dcc08e20f8a01166eab5ebcf7df5cb6b'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-002/metadata.json': {'bytes': 2286, 'sha256': 'f02d067754186ebab919d278bc5e43f2c0d824e99952c664d36707cef3a45f3b'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/epoch-002/state.pt': {'bytes': 28921503, 'sha256': 'ad432d11a230c38aac32d56ace419e5c9cf212e035a1dbbd7c9851ee387041d3'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/refit-batch-epoch-3-step-1254-7581baf4e5594c54b52a6ac7df24daa9/COMPLETE.json': {'bytes': 183, 'sha256': 'e1963d6762be8652de75ef7ebe2152b2aa016ebbe7935c70b583aab4695425f1'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/refit-batch-epoch-3-step-1254-7581baf4e5594c54b52a6ac7df24daa9/metadata.json': {'bytes': 2286, 'sha256': '8b01b9647fa142b1664393dc9dc4c306907e94a8048d6752a04b46d754c15fbf'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/refit-batch-epoch-3-step-1254-7581baf4e5594c54b52a6ac7df24daa9/state.pt': {'bytes': 28930975, 'sha256': '13ff97d60efabef65b84b03d178d8c2a24a4c8975febc687d1f9eca895c138e6'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-001-884e5f7f62764b92934e6a3065f66cf5-progress.jsonl': {'bytes': 66347, 'sha256': '17ad8fdd6af3523e27b438f2618cce1bc54f90f36cb568dc3cbfd0b4af98a931'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-001-884e5f7f62764b92934e6a3065f66cf5.json': {'bytes': 31246, 'sha256': '909f8bcf41b9f3f9c53d529e3005c35354877d29c4438696bc7bbc8c25b67a28'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-002-884e5f7f62764b92934e6a3065f66cf5-progress.jsonl': {'bytes': 21348, 'sha256': 'deb0542fc815356d71fc365ecd14256d71b9c22f92140c8788ad44d36bd6ab96'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-002-884e5f7f62764b92934e6a3065f66cf5.json': {'bytes': 31076, 'sha256': '737c42dfc8de5e86d29d5916fb0c775d1ae4fd426a031e41de877fb049a54126'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-002-a5a52d307d314bd585f8250721c29e7e-progress.jsonl': {'bytes': 45016, 'sha256': '6bc688a27ed4b236415c9d34280f885cd6a3ee2147d98518ef582b35da28c803'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-002-a5a52d307d314bd585f8250721c29e7e.json': {'bytes': 31247, 'sha256': '23471fc38acb2dbd58396477ed1e82c6bc3935ae0f1b6dc067ad9ad08585a8b0'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-003-a5a52d307d314bd585f8250721c29e7e-progress.jsonl': {'bytes': 37243, 'sha256': '1989273113f116030f8123124ea795730f9b5fba343bc99731a2b6d7e0912595'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/epoch-003-a5a52d307d314bd585f8250721c29e7e.json': {'bytes': 31203, 'sha256': '8916d6de1d1b91db097e25bbf303be0ba176cbf3357a4608d90eecd0627b1e62'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/partial-884e5f7f62764b92934e6a3065f66cf5.json': {'bytes': 31171, 'sha256': '093726d20eaefd3ec61a031749d83f39792707faa193d2f43b36a16a3b8f59f2'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/partial-a5a52d307d314bd585f8250721c29e7e.json': {'bytes': 62157, 'sha256': '0db7516126a5f59eaab5b001972bc7dad4fdf8ac5797929c6f443f533982288e'}, 'recovery/experiments/akt/results/junyi-a3-refit-20261114-01/run-manifest.json': {'bytes': 2055, 'sha256': '7342265521164c5415858558de414cc683e5f427fc697f6701324f6aa074de44'}}, 'source_receipt': 'experiments/akt/results/modal-junyi-a3-refit-h100-20261115-02/remote-inventory.json', 'next_device_verified': False, 'limits_vi': 'Giữ model/Adam/RNG/sampler và history. Còn học tiếp đến đủ9epoch. Phải kiểm source, runtime và thiết bị đích trước resume. Chưa upload hoặc chạy trên Colab.'})


In [ ]:
import json, subprocess, sys, time
from pathlib import Path
root=Path("/opt/kltn")
assert Path("/content/kltn-a3-transfer/bootstrap-verified.json").is_file()
venv=root/".venv"
assert not venv.exists(), "Moi truong da ton tai, can kiem truoc khi dung lai"
started=time.monotonic()
subprocess.run([sys.executable,"-m","venv","--without-pip",str(venv)],check=True,timeout=60)
log=Path("/content/kltn-a3-transfer/runtime-install.log")
with log.open("w") as output:
    subprocess.run([sys.executable,"-m","pip","--python",str(venv/"bin/python"),"install","--require-hashes","--no-deps","--only-binary=:all:","--no-cache-dir","--retries","1","--timeout","20","-r",str(root/"artifacts/akt/runtime/modal-linux-cuda.lock.txt")],stdout=output,stderr=subprocess.STDOUT,check=True,timeout=540)
probe=subprocess.run([str(venv/"bin/python"),"-B","-c","import json,torch; assert torch.cuda.is_available(); print(json.dumps(dict(torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),vram_bytes=torch.cuda.get_device_properties(0).total_memory)))"],capture_output=True,text=True,check=True,timeout=60)
report=dict(status="PASS_ISOLATED_CUDA_RUNTIME",seconds=time.monotonic()-started,device=json.loads(probe.stdout),global_install=False)
Path("/content/kltn-a3-transfer/runtime-verified.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report))


In [ ]:
from pathlib import Path
import subprocess, json
transfer=Path("/content/kltn-a3-transfer")
controller=transfer/"colab_a3_tranche.py"
controller.write_text('"""Một lượt tiếp tục A3 có hạn giờ trên T4 miễn phí, dùng worker đã đóng."""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport time\n\nROOT = Path(\'/opt/kltn\')\nTRANSFER = Path(\'/content/kltn-a3-transfer\')\n\n\ndef sha(path):\n    with path.open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef run(attempt, checkpoint, checkpoint_sha):\n    assert json.loads((TRANSFER / \'runtime-verified.json\').read_text())[\'status\'] == \'PASS_ISOLATED_CUDA_RUNTIME\'\n    worker = ROOT / \'gpu-stage-request.json\'\n    frozen = json.loads(worker.read_text())\n    assert frozen[\'variant\'] == \'a3\' and frozen[\'frozen_epochs\'] == 9\n    logical = ROOT / \'experiments/akt/results\' / frozen[\'logical_run_id\']\n    resume = (ROOT / checkpoint).resolve()\n    assert resume.is_relative_to((logical / \'checkpoints\').resolve())\n    assert sha(resume) == checkpoint_sha\n    assert not (logical / \'COMPLETE.json\').exists(), \'Refit đã hoàn tất, không chạy thêm\'\n    execution_id = f\'colab-junyi-a3-refit-20261115-{attempt:03}\'\n    operation = ROOT / \'experiments/akt/results\' / (execution_id + \'-operation\')\n    operation.mkdir(exist_ok=False)\n    request = dict(execution_id=execution_id, gpu=\'T4\', worker_request_sha256=sha(worker),\n                   approval_reference=\'Người dùng15/11 duyệt tiếp tục A3 bằng T4 Colab miễn phí và chuyển gói683MB.\',\n                   resume_checkpoint=checkpoint, resume_sha256=checkpoint_sha)\n    execution = operation / \'execution.json\'\n    execution.write_text(json.dumps(request, ensure_ascii=False, indent=2))\n    lock = TRANSFER / \'training.lock\'\n    lock.parent.mkdir(parents=True, exist_ok=True)\n    started = time.monotonic()\n    with lock.open(\'x\') as stream:\n        json.dump(dict(pid=os.getpid(), execution_id=execution_id), stream)\n    result = dict(status=\'FAILED\', execution_id=execution_id, checkpoint_input_sha256=checkpoint_sha)\n    process = None\n    try:\n        with (operation / \'console.log\').open(\'x\') as output:\n            process = subprocess.Popen([str(ROOT / \'.venv/bin/python\'), \'-u\', \'-B\',\n                str(ROOT / \'scripts/akt/run_junyi_refit_gpu_stage.py\'), \'--request\', str(worker),\n                \'--execution\', str(execution)], cwd=ROOT, stdout=output, stderr=subprocess.STDOUT,\n                start_new_session=True, env={**os.environ, \'PYTHONDONTWRITEBYTECODE\': \'1\'})\n            (operation / \'child.json\').write_text(json.dumps(dict(pid=process.pid)))\n            try:\n                code = process.wait(timeout=600)\n            except subprocess.TimeoutExpired:\n                os.killpg(process.pid, signal.SIGTERM)\n                try:\n                    process.wait(timeout=8)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGKILL)\n                    process.wait(timeout=10)\n                raise TimeoutError(\'Dừng lượt600giây; giữ checkpoint gần nhất, cần kiểm trước tiếp tục\')\n        result.update(exit_code=code)\n        assert code == 0, \'Worker lỗi, đọc console/failure trước khi tiếp tục\'\n        native = json.loads((operation / \'result.json\').read_text())\n        assert native[\'status\'] in {\'PARTIAL_JUNYI_REFIT90\', \'JUNYI_REFIT90_COMPLETE\'}, native[\'status\']\n        result.update(status=\'PASS_BOUNDED_NATIVE_WORKER\', native_status=native[\'status\'])\n    except BaseException:\n        if process is not None and process.poll() is None:\n            os.killpg(process.pid, signal.SIGTERM)\n            try:\n                process.wait(timeout=8)\n            except subprocess.TimeoutExpired:\n                os.killpg(process.pid, signal.SIGKILL)\n                process.wait(timeout=10)\n        raise\n    finally:\n        result[\'wall_seconds\'] = time.monotonic() - started\n        result[\'child_running\'] = process is not None and process.poll() is None\n        (operation / \'controller-result.json\').write_text(json.dumps(result, ensure_ascii=False, indent=2))\n        if not result[\'child_running\']:\n            lock.unlink()\n        print(json.dumps(result, ensure_ascii=False), flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--attempt\', type=int, required=True)\n    parser.add_argument(\'--checkpoint\', required=True)\n    parser.add_argument(\'--sha256\', required=True)\n    args = parser.parse_args()\n    assert args.attempt > 0\n    run(args.attempt, args.checkpoint, args.sha256)\n')
assert not (transfer/"training.lock").exists(), "Co luot dang chay, can kiem truoc"
console=transfer/"launch-001.log"
assert not console.exists(), "Luot001 da duoc tao, doc ket qua truoc khi chay lai"
with console.open("x") as output:
    process=subprocess.Popen(["/opt/kltn/.venv/bin/python","-u","-B",str(controller),"--attempt","1","--checkpoint",'experiments/akt/results/junyi-a3-refit-20261114-01/checkpoints/refit-batch-epoch-3-step-1254-7581baf4e5594c54b52a6ac7df24daa9/state.pt',"--sha256",'13ff97d60efabef65b84b03d178d8c2a24a4c8975febc687d1f9eca895c138e6'],stdout=output,stderr=subprocess.STDOUT,start_new_session=True)
print(json.dumps(dict(status="BOUNDED_T4_TRANCHE_LAUNCHED",pid=process.pid,log=str(console),maximum_worker_seconds=600)))
